# Mosquito Image Classifier — ResNet50 (Colab GPU)

Replicates Park et al. 2020 *Scientific Reports* paper using ResNet50 with pretrained ImageNet weights on the MosquitoDL dataset.

**Before running:** Runtime → Change runtime type → GPU (T4)

## 1. Verify GPU is attached

In [1]:
import tensorflow as tf
print('TF version:', tf.__version__)
print('GPUs available:', tf.config.list_physical_devices('GPU'))
assert tf.config.list_physical_devices('GPU'), 'No GPU! Runtime → Change runtime type → GPU'

TF version: 2.20.0
GPUs available: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


## 2. Mount Google Drive (where the MosquitoDL dataset lives)

Before running this: upload the MosquitoDL folder (with TrainVal/ and Test/) to your Google Drive — e.g. at `MyDrive/MosquitoDL/`.

In [2]:
from google.colab import drive
drive.mount('/content/drive')

import os
TRAIN_DIR = '/content/drive/MyDrive/MosquitoDL/TrainVal'
TEST_DIR  = '/content/drive/MyDrive/MosquitoDL/Test'

assert os.path.isdir(TRAIN_DIR), f'Not found: {TRAIN_DIR}'
assert os.path.isdir(TEST_DIR),  f'Not found: {TEST_DIR}'
print('Train classes:', sorted(os.listdir(TRAIN_DIR)))
print('Test classes: ', sorted(os.listdir(TEST_DIR)))

Mounted at /content/drive
Train classes: ['.DS_Store', 'Aedes albopictus', 'Aedes vexans', 'Anopheles sinensis', 'Culex pipiens', 'Culex tritaeniorhynchus', 'Non vectors']
Test classes:  ['.DS_Store', 'Aedes albopictus', 'Aedes vexans', 'Anopheles sinensis', 'Culex pipiens', 'Culex tritaeniorhynchus', 'Non vectors']


## 3. (Optional but recommended) Copy dataset to local Colab disk for faster I/O

Reading from Drive is slow. Copying to `/content/` once makes every epoch much faster.

In [ ]:
!cp -r "$TRAIN_DIR" /content/TrainVal
!cp -r "$TEST_DIR"  /content/Test
TRAIN_DIR = '/content/TrainVal'
TEST_DIR  = '/content/Test'
print('Copied — now reading from local disk.')

## 4. Hyperparameters (matches Park et al. 2020 Table 3, ResNet50 row)

In [ ]:
IMG_SIZE    = 224
BATCH_SIZE  = 16
EPOCHS      = 100
LR          = 1e-3        # was 7.5e-3 — lowered to reduce overfitting
LR_DECAY    = 0.75
LR_PERIOD   = 15
VAL_SPLIT   = 0.20
OUTPUT_DIR  = '/content/output_v2'
os.makedirs(OUTPUT_DIR, exist_ok=True)

## 5. Load datasets

In [4]:
from tensorflow.keras.applications.resnet50 import preprocess_input

def make_ds(directory, subset=None, split=None):
    kwargs = dict(seed=42, image_size=(IMG_SIZE, IMG_SIZE), batch_size=BATCH_SIZE)
    if subset:
        kwargs.update(validation_split=split, subset=subset)
    return tf.keras.utils.image_dataset_from_directory(directory, **kwargs)

train_ds = make_ds(TRAIN_DIR, subset='training',   split=VAL_SPLIT)
val_ds   = make_ds(TRAIN_DIR, subset='validation', split=VAL_SPLIT)
test_ds  = make_ds(TEST_DIR)

TARGET_NAMES = train_ds.class_names
NUM_CLASSES  = len(TARGET_NAMES)
print(f'Classes ({NUM_CLASSES}): {TARGET_NAMES}')

with open(f'{OUTPUT_DIR}/image_labels.txt', 'w') as f:
    f.write('\n'.join(TARGET_NAMES) + '\n')

# Normalise to [0,1]; ResNet50 preprocessing happens inside the model after augmentation
def to_float(x, y):
    return tf.cast(x, tf.float32) / 255.0, y

AUTOTUNE = tf.data.AUTOTUNE
train_ds = train_ds.map(to_float, num_parallel_calls=AUTOTUNE).prefetch(AUTOTUNE)
val_ds   = val_ds.map(to_float,   num_parallel_calls=AUTOTUNE).prefetch(AUTOTUNE)
test_ds  = test_ds.map(to_float,  num_parallel_calls=AUTOTUNE).prefetch(AUTOTUNE)

Found 2980 files belonging to 6 classes.
Using 2384 files for training.
Found 2980 files belonging to 6 classes.
Using 596 files for validation.
Found 2960 files belonging to 6 classes.
Classes (6): ['Aedes albopictus', 'Aedes vexans', 'Anopheles sinensis', 'Culex pipiens', 'Culex tritaeniorhynchus', 'Non vectors']


In [5]:
!ls -la /content

total 24
drwxr-xr-x 1 root root 4096 May 23 02:08 .
drwxr-xr-x 1 root root 4096 May 23 02:03 ..
drwxr-xr-x 4 root root 4096 May 21 13:32 .config
drwx------ 5 root root 4096 May 23 02:08 drive
drwxr-xr-x 2 root root 4096 May 23 02:08 output
drwxr-xr-x 1 root root 4096 May 21 13:32 sample_data


## 6. Build the model (ResNet50, all layers trainable)

In [6]:
from tensorflow.keras.applications import ResNet50
from tensorflow.keras.applications.resnet50 import preprocess_input
from tensorflow.keras import layers, models, optimizers

# Augmentation matched to paper (Test set is pre-augmented)
augmentation = tf.keras.Sequential([
    layers.RandomFlip('horizontal'),
    layers.RandomRotation(1.0),         # full 360°
    layers.RandomZoom(0.15),            # ±15%
    layers.RandomContrast(0.2),         # ±20%
    layers.RandomBrightness(0.1),       # ±10%
    layers.Lambda(lambda x: tf.image.random_hue(x, 0.1)),
    layers.Lambda(lambda x: tf.image.random_saturation(x, 0.8, 1.2)),
], name='augmentation')

base_model = ResNet50(input_shape=(IMG_SIZE, IMG_SIZE, 3),
                     include_top=False, weights='imagenet')
base_model.trainable = True

inputs  = tf.keras.Input(shape=(IMG_SIZE, IMG_SIZE, 3))           # input in [0, 1]
x       = augmentation(inputs)
x       = layers.Lambda(lambda v: preprocess_input(v * 255.0))(x) # apply ResNet50 preprocessing
x       = base_model(x)
x       = layers.GlobalAveragePooling2D()(x)
x       = layers.Dropout(0.5)(x)
outputs = layers.Dense(NUM_CLASSES, activation='softmax')(x)
model   = models.Model(inputs, outputs)

model.compile(
    optimizer=optimizers.Adam(learning_rate=LR, beta_1=0.9, beta_2=0.999),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy'],
)
model.summary()

94765736/94765736 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step


Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_1 (InputLayer)      │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ resnet50 (Functional)           │ (None, 7, 7, 2048)     │    23,587,712 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ (None, 2048)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 6)              │        12,294 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 23,600,006 (90.03 MB)

 Trainable params: 23,546,886 (89.82 MB)

 Non-trainable params: 53,120 (207.50 KB)

## 7. Train

On a Colab T4 GPU, expect roughly 10-15 minutes for 100 epochs.

In [7]:
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping, LearningRateScheduler

def lr_schedule(epoch, lr):
    if epoch > 0 and epoch % LR_PERIOD == 0:
        return lr * LR_DECAY
    return lr

callbacks = [
    ModelCheckpoint(f'{OUTPUT_DIR}/best_image_classifier.keras',
                    save_best_only=True, monitor='val_accuracy'),
    EarlyStopping(patience=20, restore_best_weights=True, monitor='val_accuracy'),
    LearningRateScheduler(lr_schedule, verbose=0),
]

history = model.fit(train_ds, validation_data=val_ds,
                   epochs=EPOCHS, callbacks=callbacks)

Epoch 1/100
149/149 ━━━━━━━━━━━━━━━━━━━━ 608s 4s/step - accuracy: 0.2836 - loss: 2.0252 - val_accuracy: 0.1393 - val_loss: 201762.2656 - learning_rate: 0.0075
Epoch 2/100
149/149 ━━━━━━━━━━━━━━━━━━━━ 27s 181ms/step - accuracy: 0.3456 - loss: 1.4995 - val_accuracy: 0.1393 - val_loss: 238.9436 - learning_rate: 0.0075
Epoch 3/100
149/149 ━━━━━━━━━━━━━━━━━━━━ 44s 203ms/step - accuracy: 0.3586 - loss: 1.5029 - val_accuracy: 0.2785 - val_loss: 3.9322 - learning_rate: 0.0075
Epoch 4/100
149/149 ━━━━━━━━━━━━━━━━━━━━ 31s 205ms/step - accuracy: 0.4316 - loss: 1.4099 - val_accuracy: 0.4597 - val_loss: 1.3764 - learning_rate: 0.0075
Epoch 5/100
149/149 ━━━━━━━━━━━━━━━━━━━━ 28s 188ms/step - accuracy: 0.4916 - loss: 1.2638 - val_accuracy: 0.2701 - val_loss: 1.6393 - learning_rate: 0.0075
Epoch 6/100
149/149 ━━━━━━━━━━━━━━━━━━━━ 28s 188ms/step - accuracy: 0.5893 - loss: 1.0354 - val_accuracy: 0.2433 - val_loss: 18.0634 - learning_rate: 0.0075
Epoch 7/100
149/149 ━━━━━━━━━━━━━━━━━━━━ 28s 187ms/step - 

## 8. Evaluate

In [ ]:
import numpy as np
from sklearn.metrics import classification_report, confusion_matrix
import seaborn as sn, pandas as pd
import matplotlib.pyplot as plt

results = model.evaluate(test_ds, verbose=0)
print(f'Test accuracy: {results[1]:.4f}  Loss: {results[0]:.4f}')

y_true = np.concatenate([y.numpy() for _, y in test_ds])
y_pred = np.argmax(model.predict(test_ds), axis=1)

print(classification_report(y_true, y_pred, target_names=TARGET_NAMES,
                            labels=np.arange(NUM_CLASSES)))

cm = confusion_matrix(y_true, y_pred, labels=np.arange(NUM_CLASSES))
df_cm = pd.DataFrame(cm, index=TARGET_NAMES, columns=TARGET_NAMES)
plt.figure(figsize=(10, 8))
sn.heatmap(df_cm, annot=True, fmt='d')
plt.title('ResNet50 — Confusion Matrix')
plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/confusion_matrix.png')
plt.show()

plt.figure(figsize=(10, 4))
plt.plot(history.history['accuracy'],     label='train')
plt.plot(history.history['val_accuracy'], label='val')
plt.xlabel('Epoch'); plt.ylabel('Accuracy'); plt.legend()
plt.title('ResNet50 Training')
plt.savefig(f'{OUTPUT_DIR}/training_curve.png')
plt.show()

## 9. Export — TFLite (float + quantized) ready for Pi

In [ ]:
model.save(f'{OUTPUT_DIR}/image_classifier_final.keras')

converter = tf.lite.TFLiteConverter.from_keras_model(model)
tflite_model = converter.convert()
with open(f'{OUTPUT_DIR}/image_classifier.tflite', 'wb') as f:
    f.write(tflite_model)
print(f'Float TFLite: {len(tflite_model)/1e6:.1f} MB')

converter.optimizations = [tf.lite.Optimize.DEFAULT]
tflite_quant = converter.convert()
with open(f'{OUTPUT_DIR}/image_classifier_quantized.tflite', 'wb') as f:
    f.write(tflite_quant)
print(f'Quantized TFLite: {len(tflite_quant)/1e6:.1f} MB')

## 10. Download outputs to your Mac

Runs the download dialog in the browser, OR save back to Drive.

In [ ]:
from google.colab import files
for name in ['image_classifier.tflite', 'image_classifier_quantized.tflite',
             'image_labels.txt', 'confusion_matrix.png', 'training_curve.png']:
    files.download(f'{OUTPUT_DIR}/{name}')

In [ ]:
# Alternative: copy everything back to Drive so it survives the session
!cp -r /content/output /content/drive/MyDrive/MosquitoDL_results